# 05 — Grad-CAM: fixed target layer, best model per architecture, quantitative localisation, all seeds

**Revision (reviewer Q5).** Every chosen configuration is now explained for **every available seed** (5 × 6 models,
≈ 1 GPU-h on a T4). Summaries report mean ± SD across seeds, and the attention effects on localisation
(ResNet50 SE/CBAM − Baseline) use a paired bootstrap that resamples test images (stratified, shared by all models)
and seeds, as in notebook 03. Example figures and class-mean CAMs still use `PREFERRED_SEED` only.

**Paper section fed:** IV-D (and the abstract's "Grad-CAM confirmed…" sentence). GPU recommended (~10–20 min).

**Target layer (state this in Methods).** The feature map entering global average pooling, i.e. the **output of
the attention module** (identity for baselines): ResNet50 `layer4`→attn, DenseNet121 `features`+ReLU→attn,
MobileNetV2 `features[18]`→attn, Custom CNN block-5→attn; 7×7 for all models. The original figures hooked
ResNet50 `layer4`, i.e. *before* SE/CBAM, so CBAM's spatial gate was not in the activations. Set
`TARGET = 'pre_attention'` to reproduce that choice for comparison.

**Quantitative checks (whole test set, not 4 hand-picked images).** Lung, diaphragm and heart masks come from
torchxrayvision's PSPNet (ChestX-Det anatomy model). Per image and model:
* `energy_in_lung` — fraction of Grad-CAM mass inside the lung masks; `lift` = that ÷ lung-area fraction (1 = chance);
* `pointing_hit` — CAM maximum falls inside the (slightly dilated) lung mask;
* `energy_below_lungs` — mass in rows below the lowest lung pixel (diaphragm/abdomen), `energy_diaphragm`;
* `energy_border` — mass in the outer 10% frame (labels, collimation edges);
* **Normal/Pneumonia overlap** — for every Normal and Pneumonia test image, the CAM for class Normal vs the CAM for
  class Pneumonia: Pearson r and IoU of the top-20% pixels; plus the IoU of class-mean CAMs.

**Mask QC is mandatory.** PSPNet was trained on adult radiographs; much of the Normal/Pneumonia data looks
paediatric. The QC grid must be inspected and the exclusion rule reported.

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# SHARED CODE — identical copy in notebooks 02, 04 and 05.
# Re-implements the 24 original notebooks in one place. Architectures, freezing,
# learning rates, loss, augmentation and schedule match the originals exactly;
# deviations are explicit, named options (see ORIGINAL-PIPELINE NOTES below).
# ═════════════════════════════════════════════════════════════════════════════
import os, re, json, time, math, random, hashlib, glob, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image

CLASS_NAMES = ['COVID19', 'NORMAL', 'PNEUMONIA', 'TURBERCULOSIS']  # folder names (dataset spells TB this way)
NUM_CLASSES = 4
IMG_EXTS = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

# ── ORIGINAL-PIPELINE NOTES (verified by reading the 20 shared notebooks) ──────
# * ResNet50: layer4 fine-tuned @1e-4; attention + fc @1e-3.
# * DenseNet121: denseblock4 + norm5 @1e-4; ReLU -> [attn] -> GAP -> fc @1e-3.
# * MobileNetV2: features[18] (final 1x1 conv 320->1280) @1e-4; [attn] + fc @1e-3.
#   The ORIGINAL BASELINE kept torchvision's Dropout(0.2) before fc, while the
#   SE/CBAM variants used no dropout. Replicated by default (head_dropout=None);
#   pass opts={'head_dropout': 0.0} to remove the confound.
# * Custom CNN: all layers @1e-3 from scratch; head 512->256->ReLU->Dropout(0.5)->4.
# * "Frozen" layers have requires_grad=False but stay in train() mode, so their
#   BatchNorm running statistics still update during training (replicated).
# * SE/CBAM hidden width = max(C//16, 4). Linear vs 1x1-conv MLP forms used in
#   different originals are mathematically equivalent.

# ─────────────────────────────── attention blocks ─────────────────────────────
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 4)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(nn.Linear(channels, hidden, bias=False), nn.ReLU(inplace=True),
                                nn.Linear(hidden, channels, bias=False), nn.Sigmoid())
    def forward(self, x):
        b, c, _, _ = x.size()
        return x * self.fc(self.avg_pool(x).view(b, c)).view(b, c, 1, 1)

class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 4)
        self.shared_mlp = nn.Sequential(nn.Linear(channels, hidden, bias=False), nn.ReLU(inplace=True),
                                        nn.Linear(hidden, channels, bias=False))
        self.avg_pool, self.max_pool = nn.AdaptiveAvgPool2d(1), nn.AdaptiveMaxPool2d(1)
    def forward(self, x):
        b, c, _, _ = x.size()
        s = self.shared_mlp(self.avg_pool(x).view(b, c)) + self.shared_mlp(self.max_pool(x).view(b, c))
        return x * torch.sigmoid(s).view(b, c, 1, 1)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
    def forward(self, x):
        m = torch.cat([x.mean(1, keepdim=True), x.max(1, keepdim=True)[0]], 1)
        return x * torch.sigmoid(self.conv(m))

class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.channel_attention = ChannelAttention(channels, reduction)
        self.spatial_attention = SpatialAttention(kernel_size)
    def forward(self, x):
        return self.spatial_attention(self.channel_attention(x))

def make_attention(kind, channels):
    if kind == 'none': return nn.Identity()
    if kind == 'se':   return SEBlock(channels)
    if kind == 'cbam': return CBAM(channels)
    raise ValueError(kind)

# ─────────────────────────────── unified network ──────────────────────────────
class XRVInput(nn.Module):
    """For torchxrayvision (CXR-pretrained) weights: undo the ImageNet normalisation of channel 0
    and map [0, 1] intensities to the [-1024, 1024] single-channel range those weights expect.
    Keeps the data pipeline identical for every model."""
    def forward(self, x):
        return (x[:, :1] * STD[0] + MEAN[0]) * 2048.0 - 1024.0

class ReLUWrap(nn.Module):           # DenseNet's functional ReLU after features
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x): return F.relu(self.m(x))

def custom_cnn_features():
    layers, cin = [], 3
    for cout in [32, 64, 128, 256, 512]:
        layers += [nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True), nn.MaxPool2d(2, 2)]
        cin = cout
    return nn.Sequential(*layers)

class CXRNet(nn.Module):
    """body -> attn -> GAP -> head.  `attn` output (== input to GAP) is the
    uniform Grad-CAM target for all 12 configurations."""
    def __init__(self, arch, attention='none', pretrained=True, unfreeze='last', head_dropout=None,
                 pretrain='imagenet', placement='single'):
        super().__init__()
        self.arch, self.attention, self.placement = arch, attention, placement
        extra = []                                   # attention modules besides the pre-pool one
        if placement not in ('single', 'multi'): raise ValueError(placement)
        if placement == 'multi' and (arch != 'resnet50' or attention == 'none'):
            raise ValueError('multi-depth placement is implemented for ResNet50 with SE/CBAM only')
        if arch == 'resnet50':
            b = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None)
            l3, l4 = b.layer3, b.layer4
            if placement == 'multi':                 # canonical SE-/CBAM-ResNet position: inside the residual branch
                for blk in list(b.layer3) + list(b.layer4):          # of every bottleneck of layer3 (6) and layer4 (3),
                    m = make_attention(attention, blk.bn3.num_features)  # after bn3, before the identity addition,
                    blk.bn3 = nn.Sequential(blk.bn3, m); extra.append(m)  # so the pretrained identity path is untouched;
                                                                         # + the pre-pool self.attn = 10 modules
            self.body = nn.Sequential(nn.Sequential(b.conv1, b.bn1, b.relu, b.maxpool), b.layer1, b.layer2, l3, l4)
            C = 2048
            ft = {'last': [self.body[4]], 'last2': [self.body[3], self.body[4]]}[unfreeze]
        elif arch == 'densenet121':
            if pretrain == 'xrv':                    # CXR-domain pretraining (torchxrayvision, 1-channel input)
                try:
                    import torchxrayvision as xrv
                except ImportError:                  # not preinstalled on Kaggle (needs Internet on)
                    import subprocess, sys
                    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torchxrayvision'])
                    import torchxrayvision as xrv
                b = xrv.models.DenseNet(weights='densenet121-res224-all' if pretrained else None)
                f = b.features; self.body = nn.Sequential(XRVInput(), ReLUWrap(f))
            elif pretrain == 'imagenet':
                b = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None)
                f = b.features; self.body = ReLUWrap(f)
            else:
                raise ValueError(pretrain)
            C = 1024
            ft = {'last': [f.denseblock4, f.norm5],
                  'last2': [f.denseblock3, f.transition3, f.denseblock4, f.norm5]}[unfreeze]
        elif arch == 'mobilenetv2':
            if pretrain != 'imagenet': raise ValueError('CXR pretraining implemented for DenseNet121 only')
            b = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1 if pretrained else None)
            self.body = b.features; C = 1280
            ft = {'last': [b.features[18]], 'last2': [b.features[17], b.features[18]]}[unfreeze]
        elif arch == 'customcnn':
            self.body = custom_cnn_features(); C = 512; ft = None
        else:
            raise ValueError(arch)
        self.attn = make_attention(attention, C)
        self.pool = nn.AdaptiveAvgPool2d(1)
        if arch == 'customcnn':
            self.head = nn.Sequential(nn.Linear(C, 256), nn.ReLU(inplace=True), nn.Dropout(0.5), nn.Linear(256, NUM_CLASSES))
        else:
            if head_dropout is None:   # replicate originals
                head_dropout = 0.2 if (arch == 'mobilenetv2' and attention == 'none') else 0.0
            self.head = nn.Sequential(nn.Dropout(head_dropout), nn.Linear(C, NUM_CLASSES)) if head_dropout > 0 \
                        else nn.Sequential(nn.Linear(C, NUM_CLASSES))
        self.channels = C
        # freezing
        if ft is not None:
            for p in self.body.parameters(): p.requires_grad = False
            for m in ft:
                for p in m.parameters(): p.requires_grad = True
        self._extra_attn = extra                     # plain list: not registered twice
        for m in extra:
            for p in m.parameters(): p.requires_grad = True
        self._ft_modules = ft

    def attention_parameters(self):
        ps = list(self.attn.parameters())
        for m in self._extra_attn: ps += list(m.parameters())
        return ps

    def forward(self, x):
        x = self.attn(self.body(x))
        return self.head(torch.flatten(self.pool(x), 1))

    def param_groups(self, lr_ft=1e-4, lr_new=1e-3, lr_attn=None, lr_scratch=1e-3):
        if self.arch == 'customcnn':
            return [{'params': list(self.parameters()), 'lr': lr_scratch, 'name': 'all'}]
        lr_attn = lr_new if lr_attn is None else lr_attn
        att = self.attention_parameters(); att_ids = {id(p) for p in att}
        g = [{'params': [p for m in self._ft_modules for p in m.parameters() if id(p) not in att_ids],
              'lr': lr_ft, 'name': 'finetuned_block'}]
        if self.attention != 'none':
            g.append({'params': att, 'lr': lr_attn, 'name': 'attention'})
        g.append({'params': list(self.head.parameters()), 'lr': lr_new, 'name': 'head'})
        return g

def count_params(model):
    tot = sum(p.numel() for p in model.parameters())
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    att = sum(p.numel() for p in model.attention_parameters())
    return {'params_total': tot, 'params_trainable': tr, 'params_attention': att}

# ─────────────────────────────── data ─────────────────────────────────────────
def list_split(root, split):
    items = []
    for ci, c in enumerate(CLASS_NAMES):
        d = os.path.join(root, split, c)
        if not os.path.isdir(d): raise FileNotFoundError(d)
        for fn in sorted(os.listdir(d)):               # same order as ImageFolder
            if fn.lower().endswith(IMG_EXTS):
                items.append((os.path.join(d, fn), ci))
    return items

def load_256(path, box=None):
    """Grayscale(3) -> Resize((256,256)) of the originals, cached as 1 channel.
    Converting back with .convert('RGB') reproduces the original tensors exactly
    (verified in the smoke test). `box` = (x0, y0, x1, y1) in [0, 1] crops the image first
    (lung-crop preprocessing; boxes from notebook 07)."""
    img = Image.open(path).convert('RGB').convert('L')   # == ImageFolder loader + Grayscale
    if box is not None:
        W, H = img.size; x0, y0, x1, y1 = box
        img = img.crop((int(round(x0 * W)), int(round(y0 * H)), int(round(x1 * W)), int(round(y1 * H))))
    img = img.resize((256, 256), Image.BILINEAR)
    return np.asarray(img, dtype=np.uint8)

_CACHE = {}
def cached_arrays(items, tag, verbose=True, boxes=None):
    key = (tag, len(items), items[0][0] if items else '')
    if key not in _CACHE:
        t = time.time()
        _CACHE[key] = np.stack([load_256(p, None if boxes is None else boxes.get(p)) for p, _ in items])
        if verbose: print(f'  cached {tag}: {len(items)} images in {time.time()-t:.0f}s')
    return _CACHE[key]

train_tf = transforms.Compose([
    transforms.RandomCrop(224),
    transforms.RandomAffine(degrees=5, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
eval_tf = transforms.Compose([transforms.CenterCrop(224), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
# Paper Sec. III-B oversampling augmentation (applied once per synthetic copy, fixed per copy)
oversample_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomAffine(degrees=7, translate=(0.04, 0.04), scale=(0.96, 1.04)),
    transforms.ColorJitter(brightness=0.08, contrast=0.08)])

class CXRDataset(Dataset):
    """index entries: (array_row, label, copy_id). copy_id>0 => synthetic copy that
    receives a FIXED oversampling augmentation (like an offline-generated file)."""
    def __init__(self, arrays, labels, index, tf, seed=0):
        self.a, self.y, self.index, self.tf, self.seed = arrays, labels, index, tf, seed
    def __len__(self): return len(self.index)
    def __getitem__(self, i):
        r, y, copy_id = self.index[i]
        img = Image.fromarray(self.a[r]).convert('RGB')
        if copy_id > 0:
            with torch.random.fork_rng(devices=[]):
                torch.manual_seed((self.seed * 1_000_003 + r * 1009 + copy_id) % (2**31))
                img = oversample_tf(img)
        return self.tf(img), y

def oversample_index(labels, seed):
    """Keep every original image; add fixed-augmented copies of minority classes
    until each class matches the majority count (no downsampling)."""
    labels = np.asarray(labels); rng = np.random.RandomState(seed)
    counts = np.bincount(labels, minlength=NUM_CLASSES); target = counts.max()
    idx = [(r, int(labels[r]), 0) for r in range(len(labels))]
    for c in range(NUM_CLASSES):
        rows = np.where(labels == c)[0]; need = target - len(rows); copy = 1
        while need > 0:
            order = rng.permutation(rows)
            for r in order[:need]:
                idx.append((int(r), c, copy))
            need -= min(need, len(order)); copy += 1
    return idx

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

def _worker_init(wid):
    s = torch.initial_seed() % 2**32; np.random.seed(s); random.seed(s)

def patient_key(path):
    """Patient identifier recoverable from the filename (Kermany-style Pneumonia/Normal names);
    COVID-19 and Tuberculosis filenames carry none, so each image is its own group."""
    fn = os.path.basename(path)
    m = re.match(r'person(\d+)_', fn)
    if m: return 'P' + m.group(1)
    m = re.match(r'(NORMAL2-)?IM-(\d+)-', fn)
    if m: return ('N2-' if m.group(1) else 'N-') + m.group(2)
    return 'F:' + fn

def grouped_val_split(items, arrays, frac=0.15, seed=12345):
    """Hold out ~frac of each class as validation, keeping patients and pixel-identical duplicates together.
    The split is fixed (independent of the training seed). Returns (train_rows, val_rows)."""
    parent = {}
    def find(k):
        while parent.setdefault(k, k) != k: parent[k] = parent[parent[k]]; k = parent[k]
        return k
    keys = [patient_key(p) for p, _ in items]; first_with_hash = {}
    for r, k in enumerate(keys):
        h = hashlib.md5(arrays[r].tobytes()).hexdigest()
        if h in first_with_hash: parent[find(k)] = find(first_with_hash[h])
        else: first_with_hash[h] = k
    groups = {}
    for r, k in enumerate(keys): groups.setdefault(find(k), []).append(r)
    labels = np.array([y for _, y in items]); rng = np.random.RandomState(seed); val = set()
    for c in range(NUM_CLASSES):
        gs = sorted({find(keys[r]) for r in np.where(labels == c)[0]}); rng.shuffle(gs)
        target, n = frac * (labels == c).sum(), 0
        for g in gs:
            if n >= target: break
            rows = [r for r in groups[g] if r not in val]; val.update(groups[g]); n += sum(labels[r] == c for r in rows)
    val_rows = sorted(val); tr_rows = [r for r in range(len(items)) if r not in val]
    return tr_rows, val_rows

# ─────────────────────────────── metrics ──────────────────────────────────────
def confusion(y, p, k=NUM_CLASSES):
    return np.bincount(np.asarray(y) * k + np.asarray(p), minlength=k * k).reshape(k, k)

def metrics_from_cm(cm):
    tp = np.diag(cm).astype(float); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp; tn = cm.sum() - tp - fp - fn
    prec = np.divide(tp, tp + fp, out=np.zeros_like(tp), where=(tp + fp) > 0)
    rec = np.divide(tp, tp + fn, out=np.zeros_like(tp), where=(tp + fn) > 0)
    f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros_like(tp), where=(prec + rec) > 0)
    spec = np.divide(tn, tn + fp, out=np.zeros_like(tp), where=(tn + fp) > 0)
    out = {'acc': tp.sum() / cm.sum(), 'macro_f1': f1.mean(), 'balanced_acc': rec.mean(),
           'weighted_f1': (f1 * cm.sum(1)).sum() / cm.sum()}
    for i, c in enumerate(CLASS_NAMES):
        out[f'prec_{c}'], out[f'rec_{c}'], out[f'f1_{c}'], out[f'spec_{c}'] = prec[i], rec[i], f1[i], spec[i]
    return out

def macro_f1_fast(y, p, k=NUM_CLASSES):
    return metrics_from_cm(confusion(y, p, k))['macro_f1']

In [ ]:
try:
    import torchxrayvision as xrv
except ImportError:
    os.system('pip install -q torchxrayvision'); import torchxrayvision as xrv
import matplotlib.pyplot as plt
from scipy.ndimage import binary_dilation

ORIGINAL_ROOT = '/kaggle/input/datasets/jtiptj/chest-xray-pneumoniacovid19tuberculosis'
RESULT_GLOBS  = ['/kaggle/input/**/results/*/metrics.json', '/kaggle/working/results/*/metrics.json']
OUT = '/kaggle/working/gradcam'; os.makedirs(OUT, exist_ok=True)
CONDITION = 'imbalanced'          # which training condition's models to explain
PREFERRED_SEED = 42               # seed used for example figures / class-mean CAMs (else the seed closest to the config mean)
ALL_SEEDS = True                  # True: explain every available seed of each chosen config (reviewer Q5); False: one seed
N_BOOT = 2000
TARGET = 'attention_output'       # or 'pre_attention'
MAX_PER_CLASS = None              # e.g. 100 to subsample; None = full test set (771)
TOP_FRAC = 0.20
QC_LUNG_FRAC = (0.08, 0.70)       # exclude images whose lung-mask area fraction is outside this range
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def find_dataset_root(preferred):
    """Use `preferred` if valid; otherwise search /kaggle/input for a folder with test/<4 classes>.
    Prefers the ORIGINAL dataset (train/PNEUMONIA has ~3,875 images) over the balanced copy (2,000);
    the test folders are identical in both, so either gives the same Grad-CAM inputs."""
    ok = lambda r: all(os.path.isdir(os.path.join(r, 'test', c)) for c in CLASS_NAMES)
    if ok(preferred): return preferred
    cands = sorted({os.path.dirname(os.path.dirname(p)) for p in glob.glob('/kaggle/input/**/test/TURBERCULOSIS', recursive=True)})
    cands = [c for c in cands if ok(c)]
    if not cands:
        raise FileNotFoundError('No folder with test/{COVID19,NORMAL,PNEUMONIA,TURBERCULOSIS} under /kaggle/input. '
                                'Attach the chest X-ray dataset via Add Input → Datasets (jtiptj/chest-xray-pneumoniacovid19tuberculosis).')
    n_pneu = lambda c: len(os.listdir(os.path.join(c, 'train', 'PNEUMONIA'))) if os.path.isdir(os.path.join(c, 'train', 'PNEUMONIA')) else 0
    root = max(cands, key=n_pneu)
    print(f'ORIGINAL_ROOT not found at {preferred!r}; using {root!r} (candidates: {cands})')
    return root
ORIGINAL_ROOT = find_dataset_root(ORIGINAL_ROOT)
print('dataset root:', ORIGINAL_ROOT)
ARCH_LABEL = {'resnet50': 'ResNet50', 'densenet121': 'DenseNet121', 'mobilenetv2': 'MobileNetV2', 'customcnn': 'Custom CNN'}
ATT_LABEL = {'none': 'Baseline', 'se': 'SE', 'cbam': 'CBAM'}

In [ ]:
# ── choose models: best config per architecture + all three ResNet50 variants ──
recs = []
for g in RESULT_GLOBS:
    for f in glob.glob(g, recursive=True):
        m = json.load(open(f))
        if m.get('status') == 'done' and not m.get('opts') and m['condition'] == CONDITION \
                and os.path.exists(os.path.join(os.path.dirname(f), 'best.pth')):
            m['dir'] = os.path.dirname(f); m['f1'] = m['test_best']['macro_f1']; recs.append(m)
R = pd.DataFrame(recs).drop_duplicates('run_id')
assert len(R), 'no finished runs with saved best.pth for this condition'
cfg_mean = R.groupby(['arch', 'attention']).f1.mean()
chosen = {}
for a in ARCH_LABEL:
    if a in cfg_mean.index.get_level_values(0): chosen[(a, cfg_mean[a].idxmax())] = 'best of architecture'
for t in ATT_LABEL:
    if ('resnet50', t) in cfg_mean.index: chosen.setdefault(('resnet50', t), 'ResNet50 family (Figs 4–6)')
SEL = []
for (a, t), why in chosen.items():
    sub = R[(R.arch == a) & (R.attention == t)].drop_duplicates('seed').sort_values('seed')
    rep_row = sub[sub.seed == PREFERRED_SEED].iloc[0] if (sub.seed == PREFERRED_SEED).any() \
              else sub.iloc[(sub.f1 - sub.f1.mean()).abs().argmin()]
    for _, row in (sub.iterrows() if ALL_SEEDS else [(None, rep_row)]):
        SEL.append(dict(arch=a, attention=t, why=why, seed=int(row.seed), example=int(row.seed) == int(rep_row.seed),
                        dir=row.dir, test_f1=row.f1, cfg_mean_f1=cfg_mean[(a, t)]))
SEL = pd.DataFrame(SEL)
print(f'{len(SEL)} model checkpoints to explain ({SEL.groupby(["arch", "attention"]).size().to_dict()})')
display(SEL.drop(columns='dir'))

In [ ]:
# ── test images + anatomy masks (aligned to the 224 centre crop) ──────────────
items = list_split(ORIGINAL_ROOT, 'test')
if MAX_PER_CLASS:
    rng = np.random.RandomState(0); keep = []
    for c in range(NUM_CLASSES):
        ids = [i for i, (_, y) in enumerate(items) if y == c]; keep += sorted(rng.choice(ids, min(MAX_PER_CLASS, len(ids)), replace=False))
    items = [items[i] for i in sorted(keep)]
A = cached_arrays(items, 'test_gc'); Yt = np.array([y for _, y in items])
seg = xrv.baseline_models.chestx_det.PSPNet().to(device).eval()
T_IDX = {n: seg.targets.index(n) for n in ['Left Lung', 'Right Lung', 'Facies Diaphragmatica', 'Heart']}
def crop224(m): return m[16:240, 16:240]
masks = {k: [] for k in ['lung', 'diaphragm', 'heart']}
with torch.no_grad():
    for i in range(0, len(A), 8):
        x = np.stack([np.asarray(Image.fromarray(a).resize((512, 512), Image.BILINEAR), dtype=np.float32) for a in A[i:i + 8]])
        x = torch.from_numpy(xrv.datasets.normalize(x, 255))[:, None].to(device)
        pr = torch.sigmoid(seg(x)) > 0.5
        pr = F.interpolate(pr.float(), size=(256, 256), mode='nearest').cpu().numpy().astype(bool)
        for p in pr:
            masks['lung'].append(crop224(p[T_IDX['Left Lung']] | p[T_IDX['Right Lung']]))
            masks['diaphragm'].append(crop224(p[T_IDX['Facies Diaphragmatica']])); masks['heart'].append(crop224(p[T_IDX['Heart']]))
masks = {k: np.stack(v) for k, v in masks.items()}
lung_frac = masks['lung'].mean((1, 2))
qc_ok = (lung_frac >= QC_LUNG_FRAC[0]) & (lung_frac <= QC_LUNG_FRAC[1])
print('mask QC pass by class:', {CLASS_NAMES[c]: f'{qc_ok[Yt == c].sum()}/{(Yt == c).sum()}' for c in range(NUM_CLASSES)})
if qc_ok.mean() < 0.8:
    print(f'WARNING: only {qc_ok.mean():.0%} of images pass mask QC — inspect mask_qc.png; localisation numbers may be unreliable.')
del seg; torch.cuda.empty_cache() if device.type == 'cuda' else None

# QC grid — inspect before trusting any number below
rng = np.random.RandomState(1); show = [rng.choice(np.where(Yt == c)[0]) for c in range(NUM_CLASSES) for _ in range(4)]
fig, ax = plt.subplots(4, 4, figsize=(12, 12))
for a_, i in zip(ax.ravel(), show):
    a_.imshow(crop224(A[i]), cmap='gray'); a_.contour(masks['lung'][i], [0.5], colors='lime', linewidths=1)
    a_.contour(masks['diaphragm'][i], [0.5], colors='orange', linewidths=1)
    a_.set_title(f"{CLASS_NAMES[Yt[i]]} lung={lung_frac[i]:.2f} {'OK' if qc_ok[i] else 'EXCLUDED'}", fontsize=8); a_.axis('off')
plt.suptitle('Mask QC: lung (green), diaphragm (orange)'); plt.tight_layout(); plt.savefig(f'{OUT}/mask_qc.png', dpi=130); plt.show()

In [ ]:
# ── Grad-CAM ──────────────────────────────────────────────────────────────────
class GradCAM:
    def __init__(self, model, target):
        self.model, self.store = model, {}
        layer = model.attn if target == 'attention_output' else model.body
        layer.register_forward_hook(self._fwd)
    def _fwd(self, mod, inp, out):
        self.store['act'] = out
        out.register_hook(lambda g: self.store.__setitem__('grad', g))
    def __call__(self, x, classes):
        """x: (B,3,224,224); classes: list of class indices or 'pred'. Returns dict cls->(B,224,224), logits."""
        self.model.zero_grad(set_to_none=True)
        logits = self.model(x); out = {}
        todo = classes
        for k, c in enumerate(todo):
            idx = logits.argmax(1) if c == 'pred' else torch.full((x.size(0),), c, device=x.device)
            sel = logits.gather(1, idx[:, None]).sum()
            sel.backward(retain_graph=k < len(todo) - 1)
            act, grad = self.store['act'].detach(), self.store['grad'].detach()
            cam = F.relu((grad.mean((2, 3), keepdim=True) * act).sum(1, keepdim=True))
            out[c] = F.interpolate(cam, size=(224, 224), mode='bilinear', align_corners=False)[:, 0].cpu().numpy()
        return out, logits.detach().softmax(1).cpu().numpy()

def top_mask(c, frac=TOP_FRAC):
    return c >= np.quantile(c, 1 - frac)

def per_image_metrics(cam, i):
    s = cam.sum() + 1e-12; L = masks['lung'][i]; Ld = binary_dilation(L, iterations=4)
    rows_l = np.where(L.any(1))[0]; low = rows_l.max() + 1 if len(rows_l) else 224
    b = int(224 * 0.10); frame = np.ones_like(L); frame[b:-b, b:-b] = False
    yx = np.unravel_index(cam.argmax(), cam.shape)
    e = (cam * L).sum() / s
    return dict(energy_in_lung=e, lung_area_frac=L.mean(), lift=e / max(L.mean(), 1e-6), pointing_hit=bool(Ld[yx]),
                energy_below_lungs=cam[low:].sum() / s, energy_diaphragm=(cam * masks['diaphragm'][i]).sum() / s,
                energy_heart=(cam * masks['heart'][i]).sum() / s, energy_border=(cam * frame).sum() / s)

rows, overlap_rows, mean_cams, examples = [], [], {}, {}
ex_idx = [next(i for i in range(len(items)) if Yt[i] == c) for c in range(NUM_CLASSES)]   # first sorted file per class (as in original figs)
for k_run, (_, s) in enumerate(SEL.iterrows()):
    name = f"{ARCH_LABEL[s.arch]} + {ATT_LABEL[s.attention]}"
    print(f'  [{k_run + 1}/{len(SEL)}] {name} seed {s.seed}')
    model = CXRNet(s.arch, s.attention, pretrained=False).to(device)
    model.load_state_dict(torch.load(os.path.join(s.dir, 'best.pth'), map_location=device)); model.eval()
    gc = GradCAM(model, TARGET)
    ds = CXRDataset(A, None, [(r, int(y), 0) for r, y in enumerate(Yt)], eval_tf)
    acc = {'pred': [], 1: [], 2: []}; probs = []
    for i0 in range(0, len(ds), 16):
        x = torch.stack([ds[i][0] for i in range(i0, min(i0 + 16, len(ds)))]).to(device).requires_grad_(True)
        cams, p = gc(x, ['pred', 1, 2]); probs.append(p)
        for k in acc: acc[k].append(cams[k])
    cams = {k: np.concatenate(v) for k, v in acc.items()}; pred = np.concatenate(probs).argmax(1)
    for i in range(len(items)):
        rows.append(dict(model=name, arch=s.arch, attention=s.attention, seed=s.seed, img=i, why=s.why, true=CLASS_NAMES[Yt[i]],
                         pred=CLASS_NAMES[pred[i]], correct=pred[i] == Yt[i], qc_ok=qc_ok[i], **per_image_metrics(cams['pred'][i], i)))
        if Yt[i] in (1, 2):
            a, b = cams[1][i].ravel(), cams[2][i].ravel()
            r = np.corrcoef(a, b)[0, 1] if a.std() > 0 and b.std() > 0 else np.nan
            ta, tb = top_mask(cams[1][i]), top_mask(cams[2][i])
            overlap_rows.append(dict(model=name, seed=s.seed, img=i, true=CLASS_NAMES[Yt[i]], pred=CLASS_NAMES[pred[i]], qc_ok=qc_ok[i],
                                     pearson_r_normalCAM_vs_pneumoniaCAM=r, iou_top20=(ta & tb).sum() / max((ta | tb).sum(), 1)))
    if s.example:                                     # figures / class-mean CAMs from the representative seed only
        for c in (1, 2):
            sel = (Yt == c) & qc_ok
            if sel.any(): mean_cams[(name, c)] = np.mean([cams['pred'][i] / (cams['pred'][i].max() + 1e-12) for i in np.where(sel)[0]], 0)
        examples[name] = [(cams['pred'][i], pred[i]) for i in ex_idx]
    del model, gc; torch.cuda.empty_cache() if device.type == 'cuda' else None
G = pd.DataFrame(rows); OV = pd.DataFrame(overlap_rows)
G.to_csv(f'{OUT}/gradcam_per_image.csv', index=False); OV.to_csv(f'{OUT}/normal_pneumonia_overlap_per_image.csv', index=False)

In [ ]:
# ── summary tables (QC-passing images only; pooled over seeds — see the across-seed table below) ───────────────────────────────────
g = G[G.qc_ok]
ms = lambda s: f'{s.mean():.3f} ± {s.std():.3f}'
summ = g.groupby(['model', 'true']).agg(n=('energy_in_lung', 'size'), energy_in_lung=('energy_in_lung', ms), lift=('lift', ms),
        pointing_game=('pointing_hit', 'mean'), energy_below_lungs=('energy_below_lungs', ms),
        energy_diaphragm=('energy_diaphragm', ms), energy_border=('energy_border', ms)).reset_index()
summ.to_csv(f'{OUT}/gradcam_summary_by_class.csv', index=False); display(summ)
overall = g.groupby('model').agg(energy_in_lung=('energy_in_lung', ms), lift=('lift', ms), pointing_game=('pointing_hit', 'mean'),
        frac_images_majority_energy_outside_lung=('energy_in_lung', lambda s: (s < 0.5).mean()),
        energy_below_lungs=('energy_below_lungs', ms)).reset_index()
overall.to_csv(f'{OUT}/gradcam_summary_overall.csv', index=False); display(overall)
print('correct vs incorrect predictions (energy in lung):')
display(g.groupby(['model', 'correct']).energy_in_lung.agg(['mean', 'std', 'size']).round(3))

o = OV[OV.qc_ok]
ovs = o.groupby(['model', 'true']).agg(n=('iou_top20', 'size'), pearson_r=('pearson_r_normalCAM_vs_pneumoniaCAM', ms), iou_top20=('iou_top20', ms)).reset_index()
rows = []
for name in G.model.unique():
    if (name, 1) in mean_cams and (name, 2) in mean_cams:
        a, b = mean_cams[(name, 1)], mean_cams[(name, 2)]; ta, tb = top_mask(a), top_mask(b)
        rows.append(dict(model=name, iou_top20_classmean_Normal_vs_Pneumonia=(ta & tb).sum() / max((ta | tb).sum(), 1),
                         cosine_classmean=float((a * b).sum() / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))))
ovs = ovs.merge(pd.DataFrame(rows), on='model', how='left') if rows else ovs
ovs.to_csv(f'{OUT}/normal_pneumonia_overlap_summary.csv', index=False); display(ovs)

In [ ]:
# ── across-seed summary: per-seed means, then mean ± SD over seeds ────────────
per_seed = g.groupby(['model', 'seed']).agg(energy_in_lung=('energy_in_lung', 'mean'), lift=('lift', 'mean'),
            pointing_game=('pointing_hit', 'mean'), energy_below_lungs=('energy_below_lungs', 'mean'),
            energy_border=('energy_border', 'mean'), frac_majority_outside=('energy_in_lung', lambda s: (s < 0.5).mean())).reset_index()
per_seed.to_csv(f'{OUT}/gradcam_per_seed.csv', index=False)
msd = lambda s: f'{s.mean():.3f} ± {s.std(ddof=1):.3f}' if len(s) > 1 else f'{s.mean():.3f}'
across = per_seed.groupby('model').agg(n_seeds=('seed', 'size'), energy_in_lung=('energy_in_lung', msd), lift=('lift', msd),
            pointing_game=('pointing_game', msd), energy_below_lungs=('energy_below_lungs', msd), energy_border=('energy_border', msd),
            frac_majority_outside=('frac_majority_outside', msd),
            energy_in_lung_min=('energy_in_lung', 'min'), energy_in_lung_max=('energy_in_lung', 'max')).reset_index()
across.to_csv(f'{OUT}/gradcam_summary_across_seeds.csv', index=False); display(across)
by_class_seed = g.groupby(['model', 'true', 'seed']).energy_in_lung.mean().groupby(['model', 'true']).agg(msd).unstack()
by_class_seed.to_csv(f'{OUT}/gradcam_energy_by_class_across_seeds.csv'); display(by_class_seed)

# ── paired bootstrap: attention − baseline on localisation (images × seeds) ───
rng = np.random.default_rng(2026)
imgs = np.array(sorted(g.img.unique()), dtype=int); ycls = Yt[imgs]
assert len(imgs), 'no QC-passing images — inspect mask_qc.png / QC_LUNG_FRAC'
BOOT = np.concatenate([np.where(ycls == c)[0][rng.integers(0, (ycls == c).sum(), (N_BOOT, (ycls == c).sum()))]
                       for c in range(NUM_CLASSES) if (ycls == c).any()], axis=1)            # shared across models
def mat(name, col):
    d = g[g.model == name].assign(v=lambda x: x[col].astype(float)).pivot_table(index='seed', columns='img', values='v').reindex(columns=imgs)
    return d.index.values, d.values.astype(float)
def loc_effect(na, nb, col):
    sa, A_ = mat(na, col); sb, B_ = mat(nb, col)
    common = sorted(set(sa) & set(sb))
    if not common: return None
    D = np.stack([A_[list(sa).index(s)] - B_[list(sb).index(s)] for s in common])        # (S, n_img)
    Dimg = D[:, BOOT].mean(2)                                                            # (S, B)
    pick = rng.integers(0, len(common), (N_BOOT, len(common)))
    d = Dimg[pick, np.arange(N_BOOT)[:, None]].mean(1); pv = D.mean(1)
    return dict(delta=pv.mean(), ci_lo=np.percentile(d, 2.5), ci_hi=np.percentile(d, 97.5), n_seeds=len(common),
                seeds_positive=int((pv > 0).sum()), p_boot=min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean())))
EFF = []
base = f"{ARCH_LABEL['resnet50']} + {ATT_LABEL['none']}"
for t in ['se', 'cbam']:
    na = f"{ARCH_LABEL['resnet50']} + {ATT_LABEL[t]}"
    if na in g.model.values and base in g.model.values:
        for col in ['energy_in_lung', 'lift', 'pointing_hit', 'energy_below_lungs', 'energy_border']:
            r = loc_effect(na, base, col)
            if r: EFF.append(dict(effect=f'{ATT_LABEL[t]} − Baseline | ResNet50 | {CONDITION}', metric=col, **r))
EFF = pd.DataFrame(EFF)
if len(EFF):
    EFF['p_holm'] = np.nan
    for _, idx in EFF.groupby('metric').groups.items():
        p = EFF.loc[idx, 'p_boot'].values; o = np.argsort(p); m = len(p); adj = np.empty(m); run = 0
        for k, i in enumerate(o): run = max(run, (m - k) * p[i]); adj[i] = min(1, run)
        EFF.loc[idx, 'p_holm'] = adj
    EFF['CI_excludes_0'] = (EFF.ci_lo > 0) | (EFF.ci_hi < 0)
    EFF.to_csv(f'{OUT}/gradcam_attention_effects.csv', index=False); display(EFF.round(4))

In [ ]:
# ── figures ───────────────────────────────────────────────────────────────────
for name, ex in examples.items():
    fig, ax = plt.subplots(2, 4, figsize=(14, 7.4))
    for c, (cam, pr) in enumerate(ex):
        i = ex_idx[c]; img = crop224(A[i])
        ax[0, c].imshow(img, cmap='gray'); ax[0, c].set_title(f'True: {CLASS_NAMES[c]}', fontsize=9)
        ax[1, c].imshow(img, cmap='gray'); ax[1, c].imshow(cam / (cam.max() + 1e-12), cmap='jet', alpha=.45)
        ax[1, c].contour(masks['lung'][i], [0.5], colors='white', linewidths=.8)
        m = per_image_metrics(cam, i)
        ax[1, c].set_title(f"Pred: {CLASS_NAMES[pr]}  lung-energy {m['energy_in_lung']:.2f}", fontsize=9,
                           color='green' if pr == c else 'red')
        for a_ in ax[:, c]: a_.axis('off')
    plt.suptitle(f'Grad-CAM ({TARGET}) — {name}; white = lung mask'); plt.tight_layout()
    plt.savefig(f"{OUT}/gradcam_{name.replace(' ', '').replace('+', '_')}.png", dpi=150); plt.show()

fig, ax = plt.subplots(figsize=(max(8, 1.6 * G.model.nunique()), 4.5))
names = list(G.model.unique()); pos = 0; ticks = []
for k, name in enumerate(names):
    data = [g[(g.model == name) & (g.true == c)].energy_in_lung.values for c in CLASS_NAMES]
    ax.boxplot(data, positions=pos + np.arange(4) * 0.8, widths=.6, showfliers=False); ticks.append(pos + 1.2); pos += 4
ax.set_xticks(ticks); ax.set_xticklabels(names, rotation=20, fontsize=8)
ax.axhline(g.lung_area_frac.mean(), ls=':', c='gray', label='mean lung-area fraction (chance)')
ax.set_ylabel('fraction of Grad-CAM energy inside lungs'); ax.set_title('boxes per model: COVID / Normal / Pneumonia / TB'); ax.legend()
plt.tight_layout(); plt.savefig(f'{OUT}/energy_in_lung_boxplot.png', dpi=150); plt.show()
json.dump(dict(target=TARGET, condition=CONDITION, n_images=len(items), n_qc_pass=int(qc_ok.sum()), qc_rule=QC_LUNG_FRAC,
               models=SEL.drop(columns='dir').to_dict('records')), open(f'{OUT}/gradcam_config.json', 'w'), indent=1, default=str)